# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Oct 29, 2026 - Day 19: Hierarchical models in PyMC, and what you didn't observe

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from ipywidgets import interactive_output, FloatSlider, Checkbox, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in = 940 px fits a 13-inch MacBook at 1280 px; 7.4 in beside widget controls

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    # the plot on the left, its controls stacked on the right (the Day 12 layout)
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# Tuesday's 32 GWTC-3 binary black holes (GWOSC, CC BY 4.0; the CSV header has the provenance)
gw = pd.read_csv('data/gwtc3_bbh_masses.csv', comment='#')
mc = gw['chirp_mass_source'].to_numpy(float)                                                     # chirp mass y_j, solar masses
mc_err = ((gw['chirp_mass_source_err_lo'] + gw['chirp_mass_source_err_hi']) / 2 / 1.645).to_numpy(float)   # sigma_j: 90% half-width -> 1 sigma
dist = gw['luminosity_distance'].to_numpy(float)                                                 # Mpc
snr = gw['network_snr'].to_numpy(float)
near30 = (mc > 20) & (mc < 40)                                                                    # the events between 20 and 40

# The PyMC runs take ~5 s each; they are cached in data/day19_cache.npz. Delete the file and this cell re-runs them.
K = 2.2   # detection probability ∝ VT ∝ m^2.2, Fishbach & Holz 2017 (ApJL 851, L25), Fig. 1
def fit(y, s, kind, selection=False, log=False, seed=19):
    import pymc as pm
    N_det = len(y)
    with pm.Model():
        mu = pm.Normal('mu', np.log(20), 1) if log else pm.Normal('mu', 20, 20)                  # the population mean
        sigma_pop = pm.HalfNormal('sigma_pop', 1 if log else 20)                                    # the population spread
        if kind == 'centered':
            theta = pm.Normal('theta', mu, sigma_pop, shape=N_det)                                  # one true value per event: the plate
        else:
            eta = pm.Normal('eta', 0, 1, shape=N_det)                                               # non-centered: theta = mu + sigma_pop * eta
            theta = pm.Deterministic('theta', mu + sigma_pop * eta)
        if selection:                                                                              # p_det(theta) / alpha(mu, sigma_pop), per event
            pm.Potential('selection', K * theta.sum() - N_det * (K * mu + K**2 * sigma_pop**2 / 2))
        pm.Normal('obs', theta, s, observed=y)                                                      # the grey circle
        tr = pm.sample(1000, tune=1000, chains=4, cores=1, random_seed=seed, progressbar=False,
                       target_accept=0.9 if selection else 0.8)
    p = tr.posterior
    return dict(mu=p['mu'].values, sigma_pop=p['sigma_pop'].values, theta=p['theta'].values, div=tr.sample_stats['diverging'].values)

RUN_KEYS = ['all', 'sub_c', 'sub_nc', 'ln_naive', 'ln_sel', 'all_nc']
if os.path.exists('data/day19_cache.npz'):
    _c = np.load('data/day19_cache.npz')
    runs = {r: {k: _c[f'{r}__{k}'] for k in ['mu', 'sigma_pop', 'theta', 'div']} for r in RUN_KEYS}
else:
    runs = {'all': fit(mc, mc_err, 'centered'), 'sub_c': fit(mc[near30], mc_err[near30], 'centered'),
            'sub_nc': fit(mc[near30], mc_err[near30], 'noncentered'),
            'ln_naive': fit(np.log(mc), mc_err / mc, 'centered', log=True),
            'ln_sel': fit(np.log(mc), mc_err / mc, 'centered', selection=True, log=True),
            'all_nc': fit(mc, mc_err, 'noncentered')}

## What did partial pooling do to Tuesday's 32 black holes?

How did Lab 07 go?

<img src="images/day18_pgm_pooling.png" alt="Three graphs for 32 objects: complete pooling with one mass mu for all, no pooling with an independent theta_j per object, and partial pooling where each theta_j is drawn from a population with mean mu and width sigma_pop" style="display:block;margin:0 auto;max-height:230px;max-width:100%">

* partial pooling draws each true $\theta_j$ from $\mathcal{N}(\mu, \sigma_{\rm pop})$
* Tuesday's partial-pooling fit gave $\sigma_{\rm pop} \approx 11\,M_\odot$, the spread of the true masses
* the outliers shrank toward the mean - this is **shrinkage**
* some of you already used `PyMC` on the midterm

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Today we're going to write the same graph in `PyMC`, where any distribution can go in it.**

</div>

## The events far from the mean *and* with wide error bars got the most shrinkage

<img src="images/day18_static_pooling.png" alt="Tuesday's population demo at the posterior median of the spread: each event's measured chirp mass with its error bar next to its partially pooled value, pulled toward the population mean; right panel the posterior of the spread" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

* GW200220 (62 $M_\odot$) shrank the most - it is the farthest from the mean, with a wide error bar
* and GW200220 ends at 41 only because we assumed one Gaussian - today we try another shape

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A wide error bar alone isn't enough - GW200308 has the widest (19 $M_\odot$), but it is near the mean and moves only 34 to 25.**

</div>

## Today - are the black holes LIGO saw a fair sample?

* Tuesday's graph as `PyMC` code
* why `PyMC` sometimes warns you not to trust its samples, and what to do about it
* which mergers LIGO misses, and what that does to the masses we measure
* putting the detector into the model, and the corrected masses
* last 12 minutes: **Chayil Ewing** and **Dante Ratchev** defend Lab 07
* Lab 08 posts today: your midterm's Part 2, redrawn as a graph and refit in `PyMC`

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**By the end we'll have LIGO's detector in the graph, and a typical black-hole mass half what we measured.**

</div>

In [ ]:
# Tuesday's graph next to its PyMC code. Hover a circle (or a line of code) to see that circle's distribution
# and light up its line. Pure CSS (:has + :hover), no JavaScript (a "skip" cell)
import io, base64, html
from IPython.display import HTML
def _png(draw):
    fig, ax = plt.subplots(figsize=(4.2, 2.1)); draw(ax); ax.set_yticks([]); plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=120, facecolor='white'); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()
def _peak1(ax, v, color, label):                                   # a histogram scaled to peak 1, so a prior and a posterior share an axis
    h, e = np.histogram(v, 60); ax.stairs(h / h.max(), e, color=color, lw=2, label=label)
def _pgm_minis():
    r = runs['all']; th = r['theta'].reshape(-1, len(mc)); mu_m, sp_m = np.median(r['mu']), np.median(r['sigma_pop'])
    def mu_ax(ax):
        x = np.linspace(-20, 60, 300); ax.plot(x, np.exp(-0.5 * ((x - 20) / 20)**2), color='0.5', lw=2, label='prior N(20, 20)')
        _peak1(ax, r['mu'].ravel(), 'C0', 'posterior'); ax.set_xlabel(r'$\mu$ [$M_\odot$]'); ax.legend(fontsize=10, loc='upper left', frameon=False)
    def sp_ax(ax):
        x = np.linspace(0, 40, 300); ax.plot(x, np.exp(-0.5 * (x / 20)**2), color='0.5', lw=2, label='prior HalfNormal(20)')
        _peak1(ax, r['sigma_pop'].ravel(), 'C0', 'posterior'); ax.set_xlabel(r'$\sigma_{\rm pop}$ [$M_\odot$]'); ax.legend(fontsize=10, loc='upper left', frameon=False)
    def th_ax(ax):
        x = np.linspace(-10, 70, 300); ax.plot(x, np.exp(-0.5 * ((x - mu_m) / sp_m)**2), color='C0', lw=2, label=r'$\mathcal{N}(\mu, \sigma_{\rm pop})$ at the medians')
        ax.plot(np.median(th, 0), np.full(len(mc), 0.05), '|', color='k', ms=14, label=r'the 32 true $\theta_j$ (medians)')
        ax.set_xlabel(r'true chirp mass $\theta_j$ [$M_\odot$]'); ax.legend(fontsize=10, loc='upper left', frameon=False)
    def y_ax(ax):
        j = int(np.argmax(mc)); t = np.median(th[:, j]); x = np.linspace(20, 100, 300)
        ax.plot(x, np.exp(-0.5 * ((x - t) / mc_err[j])**2), color='0.4', lw=2, label=rf'$\mathcal{{N}}(\theta_j, \sigma_j)$, GW200220')
        ax.axvline(mc[j], color='k', lw=2, label=f'measured $y_j$ = {mc[j]:.0f}'); ax.set_xlabel(r'chirp mass [$M_\odot$]'); ax.legend(fontsize=10, loc='upper left', frameon=False)
    def s_ax(ax):
        ax.hist(mc_err, 12, color='0.6'); ax.set_xlabel(r'the 32 error bars $\sigma_j$ [$M_\odot$] (fixed, not fit)')
    return {k: _png(f) for k, f in [('mu', mu_ax), ('sp', sp_ax), ('th', th_ax), ('y', y_ax), ('s', s_ax)]}
_CODE = [(None, "with pm.Model():"),
         ('mu', "    mu = pm.Normal('mu', 20, 20)"),
         ('sp', "    sigma_pop = pm.HalfNormal('sigma_pop', 20)"),
         ('th', "    theta = pm.Normal('theta', mu, sigma_pop, shape=32)"),
         ('y', "    pm.Normal('obs', theta, mc_err, observed=mc)"),
         (None, "    trace = pm.sample(1000, tune=1000, chains=4)")]
_NODES = {'mu': (60, 45, r'μ', 'blue'), 'sp': (190, 45, 'σ<tspan font-size="13" dy="5">pop</tspan>', 'blue'), 'th': (125, 140, 'θ<tspan font-size="13" dy="5">j</tspan>', 'white'),
          'y': (125, 240, 'y<tspan font-size="13" dy="5">j</tspan>', 'grey'), 's': (235, 240, 'σ<tspan font-size="13" dy="5">j</tspan>', 'dot')}
def pgm_hover():
    minis = _pgm_minis(); k = list(_NODES)
    css = '.pgmw .pop{display:none} .pgmw .nd{cursor:pointer} .pgmw pre{font-size:15px;line-height:1.6;margin:0} .pgmw .ln{display:block;padding:0 4px}'
    for n in k:
        on = f'.pgmw:has(.n-{n}:hover, .ln-{n}:hover)'
        ln = 'y' if n == 's' else n                                     # sigma_j has no line of its own: it is mc_err in the obs line
        css += f' {on} .pop-{n}{{display:block}} {on} .ln-{ln}{{background:#fde68a}} {on} .n-{n} circle{{stroke:#f59e0b;stroke-width:5}}'
    css += ' .pgmw:has(.nd:hover, .ln:hover) .hint{display:none}'
    svg = ['<svg width="300" height="290" viewBox="0 0 300 290" font-family="sans-serif">',
           '<defs><marker id="ah19" markerWidth="10" markerHeight="8" refX="9" refY="4" orient="auto"><path d="M0,0 L10,4 L0,8 z"/></marker></defs>',
           '<rect x="70" y="95" width="215" height="185" fill="none" stroke="black"/><text x="280" y="272" font-size="15" text-anchor="end">j = 1…32</text>']
    for a, b in [('mu', 'th'), ('sp', 'th'), ('th', 'y'), ('s', 'y')]:
        (x1, y1, *_), (x2, y2, *_) = _NODES[a], _NODES[b]; L = np.hypot(x2 - x1, y2 - y1)
        r1 = 6 if a == 's' else 30
        svg.append(f'<line x1="{x1 + (x2 - x1) * r1 / L:.0f}" y1="{y1 + (y2 - y1) * r1 / L:.0f}" x2="{x2 - (x2 - x1) * 32 / L:.0f}" y2="{y2 - (y2 - y1) * 32 / L:.0f}" stroke="black" stroke-width="1.5" marker-end="url(#ah19)"/>')
    for n, (x, y, lab, kind) in _NODES.items():
        if kind == 'dot':
            svg.append(f'<g class="nd n-{n}"><circle cx="{x}" cy="{y}" r="16" fill="white" stroke="white"/><circle cx="{x}" cy="{y}" r="6" fill="black"/><text x="{x}" y="{y - 14}" text-anchor="middle" font-size="20">{lab}</text></g>')
        else:
            fill = {'blue': 'white', 'white': 'white', 'grey': '#b3b3b3'}[kind]; ec = '#1f77b4' if kind == 'blue' else 'black'
            svg.append(f'<g class="nd n-{n}"><circle cx="{x}" cy="{y}" r="30" fill="{fill}" stroke="{ec}" stroke-width="{3 if kind == "blue" else 1.5}"/><text x="{x}" y="{y + 7}" text-anchor="middle" font-size="22" font-style="italic">{lab}</text></g>')
    svg.append('</svg>')
    code_html = ''.join(f'<span class="ln{" ln-" + n if n else ""}">{html.escape(t)}</span>' for n, t in _CODE)
    pops = ''.join(f'<img class="pop pop-{n}" src="data:image/png;base64,{minis[n]}" style="width:380px">' for n in k)
    display(HTML(f'<style>{css}</style><div class="pgmw" style="display:flex;gap:24px;align-items:flex-start">'
                 f'<div>{"".join(svg)}</div><div><pre>{code_html}</pre><div style="height:200px;margin-top:10px">'
                 f'<div class="hint" style="color:#666;padding-top:60px">hover a circle or a line to see its distribution</div>{pops}</div></div></div>'))

## How do we write Tuesday's graph in `PyMC`?

In [ ]:
# Tuesday's graph and its code: hover a circle
pgm_hover()

<img src="images/day19_pgm_pooling_gw.png" alt="Partial pooling for the 32 events: blue mu and sigma_pop above a plate; inside the plate a white circle theta_j, the true chirp mass, with an arrow to the grey measured chirp mass y_j, which also has an arrow from the fixed error bar sigma_j" style="display:block;margin:0 auto;max-height:220px;max-width:100%">

```python
with pm.Model():
    mu = pm.Normal('mu', 20, 20)                              # blue circle
    sigma_pop = pm.HalfNormal('sigma_pop', 20)                # blue circle, and > 0
    theta = pm.Normal('theta', mu, sigma_pop, shape=32)       # white circle; shape=32 is the plate
    pm.Normal('obs', theta, mc_err, observed=mc)              # grey circle y_j: observed= marks the data
    trace = pm.sample(1000, tune=1000, chains=4)
```

(static copy of the hover graph, for readers on GitHub or without running the notebook)

* one line per circle, and the plate becomes `shape=32`
* `observed=` marks the grey circle: `mc` and `mc_err` hold the 32 $y_j$ and $\sigma_j$
* `HalfNormal` is a Gaussian folded at zero, since a spread can't be negative
* hover $\mu$ or $\sigma_{\rm pop}$ to see its wide grey prior and blue posterior
* `PyMC` samples all 34 parameters (32 masses, $\mu$, $\sigma_{\rm pop}$) with NUTS (Day 14's Hamiltonian sampler)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Notice that every arrow into a circle shows up as an argument in that circle's line!**

</div>

In [ ]:
# Posterior of the population and the shrinkage of each event, from the centered fit to all 32 (a "skip" cell)
from scipy.stats import gaussian_kde
def plot_all32():
    r = runs['all']
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.4, 3.6), gridspec_kw={'width_ratios': [1, 1.6]})
    m, s = r['mu'].ravel(), r['sigma_pop'].ravel()
    kde = gaussian_kde(np.vstack([m, s])); at = kde(np.vstack([m, s]))
    X, Yg = np.meshgrid(np.linspace(m.min(), m.max(), 120), np.linspace(s.min(), s.max(), 120))
    Z = kde(np.vstack([X.ravel(), Yg.ravel()])).reshape(X.shape)
    lev = [np.quantile(at, 0.05), np.quantile(at, 0.32)]                # contours holding 95% and 68% of the draws
    a1.contourf(X, Yg, Z, levels=lev + [Z.max()], colors=['#c6dbef', '#6baed6'])
    a1.contour(X, Yg, Z, levels=lev, colors='C0', linewidths=1.2)
    mm, sm = np.median(m), np.median(s)
    a1.axvline(mm, color='k', ls='--', lw=1); a1.axhline(sm, color='k', ls='--', lw=1)
    a1.plot(mm, sm, '+', color='k', ms=14, mew=2)
    a1.text(0.97, 0.95, f'medians\n$\\mu$ = {mm:.1f}\n$\\sigma_{{\\rm pop}}$ = {sm:.1f}', transform=a1.transAxes, ha='right', va='top', fontsize=11)
    a1.set_xlabel(r'$\mu$ [$M_\odot$]'); a1.set_ylabel(r'$\sigma_{\rm pop}$ [$M_\odot$]'); a1.set_title('68% and 95% of 4000 draws', fontsize=12)
    o = np.argsort(mc); th = r['theta'].reshape(-1, len(mc))
    x = np.arange(len(mc)); meas = mc[o]; post = np.median(th, 0)[o]
    a2.errorbar(x - 0.3, meas, mc_err[o], fmt='o', color='0.5', ms=4, label='as measured')
    a2.errorbar(x + 0.3, post, th.std(0)[o], fmt='s', color='C0', ms=4, label='partially pooled')
    for k in x:                                                         # before -> after: an arrow for every event that moves
        if abs(post[k] - meas[k]) > 1:
            a2.annotate('', xy=(k + 0.3, post[k]), xytext=(k - 0.3, meas[k]),
                        arrowprops=dict(arrowstyle='->', color='k', lw=1.6, shrinkA=2, shrinkB=2))
    a2.axhline(mm, color='C0', ls='--', lw=1)
    a2.set_xlabel('event, sorted by chirp mass'); a2.set_ylabel(r'chirp mass [$M_\odot$]'); a2.legend(loc='upper left', fontsize=11)
    plt.tight_layout(); plt.show()

# az.summary and the traces of the two population parameters, from the cached draws
idata_all = az.from_dict({'posterior': {k: runs['all'][k] for k in ['mu', 'sigma_pop']}})
def summary_all32():
    return az.summary(idata_all)[['mean', 'sd', 'ess_bulk', 'ess_tail', 'r_hat']]
def plot_trace_all32():
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 2.6))
    for ax, name, lab in zip(axs, ['mu', 'sigma_pop'], [r'$\mu$ [$M_\odot$]', r'$\sigma_{\rm pop}$ [$M_\odot$]']):
        v = idata_all.posterior[name].values
        for j in range(v.shape[0]):
            ax.plot(v[j], lw=0.4, alpha=0.8)
        ax.set_ylabel(lab); ax.set_xlabel('draw')
    plt.tight_layout(); plt.show()

## Sampling all 32 gives back Tuesday's spread of 11 $M_\odot$

In [ ]:
# the population posterior (left) and each event before and after pooling (right)
plot_all32()

* the shaded contours (left) hold 68% and 95% of the 4000 draws (4 chains x 1000) of $(\mu, \sigma_{\rm pop})$, and the cross marks the medians
* Tuesday's shrinkage plot (right), now drawn from samples
* `az.summary` gives $\hat R$ = 1.00 for both, and thousands of effective samples (Day 14's checks)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Because `PyMC` does the sampling, we're free to change the graph and refit it.**

</div>

## The four chains agree ($\hat R$ = 1.00, thousands of effective samples)

(not a slide: the table and the traces behind the bullet above)

In [ ]:
# az.summary of the four chains for the two population parameters, then their traces
display(summary_all32())
plot_trace_all32()

* `r_hat` is 1.00 for both, and `ess_bulk` counts thousands of effective samples - Day 14's checks 2 and 3
* the four traces overlap and wander over the same range

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Check $\hat R$ and the effective samples after every `PyMC` run!**

</div>

## Is the peak near 30 $M_\odot$ one tight population?

<img src="images/day18_gwtc3_masses.png" alt="GWTC-3 population paper Fig. 10: the inferred merger rate per unit primary mass, with peaks near 10 and 35 solar masses, and per unit mass ratio" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Abbott et al. 2023, PRX 13, 011048, Fig. 10 (CC BY 4.0). Tuesday's population-paper figure: the merger rate against the heavier mass $m_1$ (the chirp-mass peaks are in the first bullet).</div>

* in chirp mass, the population paper finds peaks at 8.3 and 27.9 $M_\odot$
* 15 of our 32 events are between 20 and 40 $M_\odot$; if they all formed the same way, their true masses should have a small spread

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Fitting just those 15 measures the spread of the peak.**

</div>

## The same model on those 15 events gives 38 divergences

```
There were 38 divergences after tuning. Increase `target_accept` or reparameterize.
```

* this is the warning `PyMC` prints, from the same code
* 38 of the 4000 draws are flagged **divergent**: the sampler's path to that draw went wrong
* raising `target_accept` (the acceptance rate NUTS aims for, 0.8 by default) shrinks every step
* to *reparameterize* is to rewrite the model so the sampler sees an easier shape
<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Some of the posterior may never have been explored.**

</div>

## A divergence is a leapfrog path that leaves the bowl

<img src="images/day14_leapfrog_vs_euler.gif" alt="Animation from Day 14: two balls start at the same point in a bowl-shaped curve. The blue leapfrog ball rolls back and forth and its energy line stays flat; the red Euler ball swings higher each pass and leaves the bowl while its energy line climbs off the plot" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Day 14 (Oct 8): the ball rolls in the bowl $-\ln p$; left the path, right its energy $H$.</div>

* NUTS rolls the ball in leapfrog steps of one size $\epsilon$, picked during `tune`
* on Day 14 the bowl had the same curvature everywhere, so one $\epsilon$ kept the energy $H$ nearly constant
* if the posterior narrows somewhere, that $\epsilon$ is too big there, and $H$ jumps
* `PyMC` stops a path whose energy error gets too big and flags it divergent

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Where the posterior narrows, there's no one right step size.**

</div>

In [ ]:
# Posterior draws of one event's true chirp mass against the spread; divergent draws in red (a "skip" cell)
def _funnel_ax(ax, key):
    r = runs[key]; j = 0
    y = r['sigma_pop'].ravel(); x = r['theta'][..., j].ravel(); bad = r['div'].ravel().astype(bool)
    if key.endswith('nc'):
        x = (x - r['mu'].ravel()) / y                                    # eta_1, the coordinate the sampler moves in
    ax.plot(x[~bad], y[~bad], '.', ms=2, color='C0', alpha=0.4, label='draws')
    pts = len(np.unique(y[bad]))
    ax.plot(x[bad], y[bad], 'o', ms=6, color='C3', label=f'divergent: {bad.sum()}' + (' (1 point)' if pts == 1 < bad.sum() else ''))
    ax.set_yscale('log'); ax.set_ylim(1e-3, 20); ax.set_ylabel(r'$\sigma_{\rm pop}$ [$M_\odot$]')
    ax.set_xlabel(r'$\eta_1 = (\theta_1 - \mu)/\sigma_{\rm pop}$' if key.endswith('nc') else r'$\theta_1$, first event [$M_\odot$]')
    ax.legend(loc='lower right', fontsize=10)
def plot_funnel():
    # the whole funnel from the non-centered draws (grey), and where the centered sampler got to (blue, red)
    fig, ax = plt.subplots(figsize=(7.4, 4.0))
    n = runs['sub_nc']; ax.plot(n['theta'][..., 0].ravel(), n['sigma_pop'].ravel(), '.', ms=2, color='0.75', label='the whole posterior')
    _funnel_ax(ax, 'sub_c'); h, l = ax.get_legend_handles_labels()
    ax.legend(h, ['the whole posterior (sampled another way)', 'draws from the code so far (centered)', l[2]], loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=9, frameon=False)
    ax.annotate('the neck: $\\sigma_{\\rm pop}$ small,\nevery $\\theta_j$ near $\\mu$', xy=(27.8, 0.02), xytext=(17.5, 0.004), fontsize=11, arrowprops=dict(arrowstyle='->'))
    ax.set_title(f'the fit to the {int(near30.sum())} near 30 never reaches the neck', fontsize=12)
    plt.tight_layout(); plt.show()
def plot_funnel_pair():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.4, 3.4), sharey=True)
    _funnel_ax(a1, 'sub_c'); _funnel_ax(a2, 'sub_nc'); a2.set_ylabel('')
    a1.set_title(r'centered: the sampler moves in $\theta_j$', fontsize=12); a2.set_title(r'non-centered: the sampler moves in $\eta_j$', fontsize=12)
    plt.tight_layout(); plt.show()

# The two ways of writing it, side by side: divergences on the 15, effective samples of sigma_pop on all 32
def plot_centered_vs_noncentered():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.4, 2.6))
    ess = []
    for key in ['all', 'all_nc']:
        ess.append(int(az.ess(az.from_dict({'posterior': {'s': runs[key]['sigma_pop']}}))['s']))
    div = [int(runs['sub_c']['div'].sum()), int(runs['sub_nc']['div'].sum())]
    for ax, v, t in [(a1, ess, r'strong data, all 32: effective samples of $\sigma_{\rm pop}$ (more is better)'), (a2, div, f'weak data, the {int(near30.sum())} near 30: divergences (fewer is better)')]:
        ax.barh(['centered', 'non-centered'], v, color=['C0', 'C1'])
        for k, val in enumerate(v):
            ax.text(val, k, f' {val}', va='center', fontsize=12)
        ax.set_title(t, fontsize=11); ax.set_xlim(0, max(v) * 1.25); ax.invert_yaxis()
    plt.tight_layout(); plt.show()

## The divergences are all in the neck of a funnel

In [ ]:
# one event's true chirp mass against the spread, for the events between 20 and 40
plot_funnel()

* each dot is one draw: the first event's true mass $\theta_1$ against the spread $\sigma_{\rm pop}$ (log scale)
* small $\sigma_{\rm pop}$ forces every $\theta_j$ close to $\mu$, so the posterior narrows to a neck - this shape is called a **funnel**
* a step size tuned for the wide top is too big for the neck, so those paths diverge (red)
* $\hat R$ = 1.02 for $\sigma_{\rm pop}$, above Day 14's 1.01

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The divergences mark the part of the posterior the sampler never explored!**

</div>

## If we rewrite the same model, one step size fits and the 38 divergences drop to 2

In [ ]:
# the same posterior in the centered coordinates (left) and the non-centered ones (right)
plot_funnel_pair()

* the answer to the peak question: its spread is under 3.6 $M_\odot$ (95%), consistent with zero
* our code so far, `theta = pm.Normal('theta', mu, sigma_pop)`, is called the **centered** form
* `theta = mu + sigma_pop * eta`, with `eta = pm.Normal('eta', 0, 1)`, is the **non-centered** form (the warning's "reparameterize")
* $\eta_j$ has width 1 whatever $\sigma_{\rm pop}$ is, so the neck on the left opens up on the right

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**It's the same model - we've only changed the coordinates the sampler steps in.**

</div>

## Write it centered when the data are strong, and non-centered when they're weak

In [ ]:
# effective samples of sigma_pop on all 32 (left) and divergences on the 15 near 30 (right)
plot_centered_vs_noncentered()

* all 32: error bars (median 2 $M_\odot$) are small next to the spread (11), and centered gives 4903 effective samples (non-centered 401)
* with strong data each $\theta_j$ is pinned by its own error bar, so the funnel moves into the non-centered form ($\eta_j$ against $\sigma_{\rm pop}$)
* the 15 near 30: error bars (3.1) are bigger than the spread, and non-centered gives 2 divergences (centered 38)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Start centered, and rewrite it non-centered if `PyMC` reports divergences!**

</div>

## But did we fit a fair sample of black holes?

<img src="images/day19_mantz_truncation_pair.png" alt="Simulated luminosity against mass for a population with a straight-line relation: left, every object; right, only the objects above a luminosity threshold, which sit well above the true line at low mass" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Simulated (Mantz et al. 2010, via Mantz &amp; Marshall 2019). Red: the true line; right: only the objects above a threshold.</div>

* I picked the 15 by measured mass, and LIGO picked all 32 before we saw them
* objects missing from the table, when you don't know how many, is called **truncation** (right: a line fit to what's left comes out too flat)
* **censoring** is different: the object is in the table, but its value is missing or only an upper limit

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**LIGO's catalog is truncated, and our graph has no arrow for how events got into it.**

</div>

## How loud a merger is depends on its chirp mass and its distance

<img src="images/day18_gw150914_fig1.png" alt="GW150914, the first detection: the strain at LIGO Hanford and Livingston over 0.2 seconds, the reconstructed waveform, the residuals, and a time-frequency map in which the signal sweeps up from 35 to 250 Hz as the two black holes spiral in" style="display:block;margin:0 auto;max-height:280px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Abbott et al. 2016, PRL 116, 061102, Fig. 1 (CC BY 3.0): GW150914 at Hanford and Livingston. Tuesday used its frequency sweep; today, its size.</div>

* to draw that arrow, we need to know which mergers LIGO can hear
* the sweep gives the chirp mass (Tuesday), and the amplitude, falling as $1/d$, sets the **luminosity distance** $d$
* the signal-to-noise ratio (SNR) grows as $\mathcal{M}^{5/6}$, with $\mathcal{M}$ the chirp mass (heavier pairs are louder)
* GWTC-3 really cuts on how often noise fakes a signal; our stand-in is an SNR of 8, combined over the detectors (the network SNR)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**LIGO only keeps a merger if its signal-to-noise is above a threshold.**

</div>

In [ ]:
# Chirp mass against luminosity distance for the 32 events, with the median reach at network SNR 8 (a "skip" cell)
# SNR of the inspiral ∝ Mc^(5/6) / d; one normalization, the median of d * SNR / 8 / Mc^(5/6) over the 32 events
def plot_reach():
    norm = np.median(dist * snr / 8 / mc ** (5 / 6))
    m = np.linspace(5, 70, 200)
    fig, ax = plt.subplots(figsize=(7.4, 3.6))
    ax.scatter(mc, dist / 1000, c='C0', s=25, label='the 32 events')
    ax.plot(m, norm * m ** (5 / 6) / 1000, 'k--', label=r'median reach at SNR 8, $\propto \mathcal{M}^{5/6}$')
    ax.set_xscale('log'); ax.set_yscale('log')
    from matplotlib.ticker import ScalarFormatter, NullFormatter
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_formatter(ScalarFormatter()); axis.set_minor_formatter(NullFormatter())
    ax.set_xticks([5, 10, 20, 40, 60]); ax.set_yticks([0.3, 1, 3, 6])
    ax.set_xlabel(r'chirp mass [$M_\odot$]'); ax.set_ylabel('luminosity distance [Gpc]'); ax.legend(loc='upper left', fontsize=11)
    plt.tight_layout(); plt.show()

## LIGO sees heavier mergers farther away

In [ ]:
# the 32 events: chirp mass against distance, and how far LIGO could reach at each mass
plot_reach()

* the reach is the distance where the SNR drops to 8 (a round threshold of ours), and it grows as $\mathcal{M}^{5/6}$
* the dashed line is the median of the 32 events' own reaches (sky position and orbit orientation change each one)
* reach cubed gives the volume, so it grows as $\mathcal{M}^{5/2}$
* far away, LIGO only detects the heavy (loud) ones - this is called **Malmquist bias** (next slide)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note that a 30 $M_\odot$ chirp mass is seen in about $27\times$ the volume of an 8.**

</div>

In [ ]:
# Malmquist and Eddington bias as widgets: our own simulations, seed 19 (a "skip" cell)
def plot_malmquist(flux_limit=-1.6, scatter=0.25):
    rng = np.random.default_rng(19)
    d = 100 * rng.uniform(0, 1, 4000) ** (1 / 3)                      # sources spread evenly through a volume, out to 100 pc
    logL = scatter * rng.normal(0, 1, 4000)                             # log luminosity, mean 0 (the typical source)
    det = logL - 2 * np.log10(d / 10) > flux_limit                      # flux ∝ L / d^2; the survey keeps everything above its limit
    fig, ax = plt.subplots(figsize=(7.4, 3.8))
    ax.plot(d[~det], logL[~det], '.', ms=3, color='0.75', label='missed')
    ax.plot(d[det], logL[det], '.', ms=3, color='C0', label='detected')
    dd = np.linspace(1, 100, 300)
    ax.plot(dd, flux_limit + 2 * np.log10(dd / 10), 'k--', lw=2, label=r'flux limit, $L_{\rm min} \propto d^2$')
    ax.axhline(0, color='0.3', lw=2, label='mean of all sources')
    edges = np.linspace(20, 100, 9); mids = (edges[1:] + edges[:-1]) / 2
    n_in = np.array([np.sum(det & (d > a) & (d < b)) for a, b in zip(edges[:-1], edges[1:])])
    mdet = np.array([logL[det & (d > a) & (d < b)].mean() if n > 0 else np.nan for n, a, b in zip(n_in, edges[:-1], edges[1:])])
    ax.plot(mids, mdet, 'o-', color='C3', lw=2.5, ms=7, label='mean of the detected')
    far = np.flatnonzero(n_in > 0)
    msg = f'farthest detected bin: mean {mdet[far[-1]]:+.2f} dex' if len(far) else 'nothing detected beyond 20 pc'
    ax.set_title(msg, fontsize=12)
    ax.set_xlim(0, 100); ax.set_ylim(-1, 1)
    ax.set_xlabel('distance [pc]'); ax.set_ylabel(r'$\log_{10} L$ (typical source = 0)')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), fontsize=9, ncol=3, frameon=False)
    plt.tight_layout(); plt.show()
def malmquist_controls():
    controls_right(plot_malmquist,
                   flux_limit=FloatSlider(value=-1.6, min=-2.4, max=-0.8, step=0.1, description='flux limit (log)'),
                   scatter=FloatSlider(value=0.25, min=0.0, max=0.5, step=0.05, description='scatter (dex)'))

def plot_eddington(scatter=0.2, steepness=1.5):
    rng = np.random.default_rng(19)
    logL_true = rng.exponential(1 / (steepness * np.log(10)), 400_000)  # counts fall as 10^(-steepness log L): steep = many more faint
    logL_obs = logL_true + scatter * rng.normal(0, 1, 400_000)          # each one measured with scatter
    thr = 1.0; kept = logL_obs > thr                                    # the survey keeps everything measured above log L = 1
    bins = np.arange(0.5, 2.21, 0.1); c = (bins[1:] + bins[:-1]) / 2
    n_true, _ = np.histogram(logL_true, bins); n_obs, _ = np.histogram(logL_obs, bins)
    fig, (ax, a2) = plt.subplots(1, 2, figsize=(7.4, 3.6), gridspec_kw={'width_ratios': [1.3, 1]})
    ax.step(c, n_true, where='mid', color='0.3', lw=2, label='true')
    ax.step(c, n_obs, where='mid', color='C0', lw=2.5, label='measured')
    ax.axvline(thr, color='k', ls='--', lw=2)
    ax.set_yscale('log'); ax.set_xlabel(r'$\log_{10} L$'); ax.set_ylabel('number per 0.1 dex')
    ax.set_title(f'measured / true above threshold: {np.median(n_obs[c > thr] / np.maximum(n_true[c > thr], 1)):.1f}', fontsize=11)
    ax.legend(loc='lower left', fontsize=9)
    fb = np.mean(logL_true[kept] < thr); bias = np.mean(logL_obs[kept] - logL_true[kept])
    hb = np.arange(0.4, 2.0, 0.05)
    a2.hist(logL_true[kept], hb, color='C0', alpha=0.6, label='truly above')
    a2.hist(logL_true[kept & (logL_true < thr)], hb, color='C1', alpha=0.9, label='truly below')
    a2.axvline(thr, color='k', ls='--', lw=2)
    a2.set_xlabel(r'true $\log_{10} L$ of the kept'); a2.set_yticks([]); a2.set_xlim(0.4, 2.0)
    a2.set_title(f'{100 * fb:.0f}% truly below; {bias:.2f} dex too bright', fontsize=11)
    a2.legend(loc='upper right', fontsize=9)
    plt.tight_layout(); plt.show()
def eddington_controls():
    controls_right(plot_eddington,
                   scatter=FloatSlider(value=0.2, min=0.0, max=0.4, step=0.05, description='scatter (dex)'),
                   steepness=FloatSlider(value=1.5, min=0.3, max=2.5, step=0.1, description='steepness'))

## Far away, a flux-limited survey keeps only the bright ones - Malmquist bias

In [ ]:
# simulated sources, evenly spread in volume, seen through one flux limit
malmquist_controls()

<img src="images/day19_static_malmquist_widget.png" alt="The Malmquist widget at its opening values: simulated sources spread evenly through a volume out to 100 pc, log luminosity against distance; the flux limit is a dashed curve rising as distance squared; detected sources in blue above it, missed ones in grey below; the mean of all sources is a flat line at zero, and the mean of the detected ones, in red, climbs to 0.46 dex in the farthest bin" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:0.8em; text-align:center; color:#666;">Our simulation (seed 19): 4000 sources, uniform in volume, one flux limit. The SDSS magnitude cut (14 < r < 21) on Days 2-4 was the same kind of bias.</div>

* every source has the same luminosity distribution (grey line: its mean)
* the faintest luminosity a flux limit lets in rises as $d^2$ (dashed), since flux = $L / 4\pi d^2$
* the mean of what's detected (red) climbs to 0.46 dex (about $3\times$) at the far edge
* slide the scatter to 0: every source is the same, the survey just stops at one distance, and the bias is gone

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The sample of detected luminosities is not representative of the whole population - even with perfect measurements.**

</div>

## LIGO's sensitive volume-time grows as (total mass)$^{2.2}$

<img src="images/day19_fishbach_holz2017_fig1.png" alt="Fishbach and Holz 2017 Fig. 1: LIGO sensitive spacetime volume VT in comoving Gpc^3 yr against binary black hole total mass from 10 to 200 solar masses, log-log, one curve per mass ratio, with a dashed power-law guide of slope 2.2" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Fishbach &amp; Holz 2017, ApJL 851, L25, Fig. 1 (arXiv:1709.08584). The legend's "VT ∝ 2.2" means VT ∝ M<sub>tot</sub><sup>2.2</sup>, $q = m_2/m_1$ is the mass ratio.</div>

* **VT**, the sensitive volume-time, is the volume in which LIGO could detect a merger, times how long it observed - more VT, more detections
* Fishbach & Holz find $VT \propto M_{\rm tot}^{2.2}$, with $M_{\rm tot} = m_1 + m_2$ - flatter than our $\mathcal{M}^{5/2}$ (why is in EXTRA)
* so a 30 is seen in about $18\times$ the VT of an 8 (the reach plot said $27\times$)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We only ever see the population weighted by VT, i.e. by $M_{\rm tot}^{2.2}$.**

</div>

In [ ]:
# A population we choose, seen through a LIGO-like detector, with sliders (a "skip" cell)
def plot_selection_sim(true_median=12.0, width=0.5, k_det=2.2):
    rng = np.random.default_rng(19)
    lnm = rng.normal(np.log(true_median), width, 100_000)              # every merger in the universe (ln chirp mass)
    w = np.exp(k_det * (lnm - lnm.mean())); w /= w.sum()               # detection probability ∝ Mc^K, as a weight on each merger
    o = np.argsort(lnm); med_det = np.exp(lnm[o][np.searchsorted(np.cumsum(w[o]), 0.5)])
    fig, ax = plt.subplots(figsize=(7.4, 3.6))
    bins = np.linspace(0, 100, 80)
    ax.hist(np.exp(lnm), bins, density=True, histtype='step', lw=2, color='0.4', label=f'all mergers, median {true_median:.0f}')
    ax.hist(np.exp(lnm), bins, weights=w, density=True, histtype='stepfilled', alpha=0.5, color='C0', label=f'detected, median {med_det:.0f}')
    ax.set_xlim(0, 100); ax.set_xlabel(r'chirp mass [$M_\odot$]'); ax.set_ylabel('density'); ax.legend(fontsize=10, loc='upper right')
    plt.tight_layout(); plt.show()
def selection_controls():
    controls_right(plot_selection_sim,
                   true_median=FloatSlider(value=12.0, min=5, max=40, step=1, description='true median'),
                   width=FloatSlider(value=0.5, min=0.2, max=0.6, step=0.05, description='width (ln)'),
                   k_det=FloatSlider(value=2.2, min=0, max=3, step=0.1, description='exponent K'))

## If LIGO favors heavy mergers, the detected sample comes out heavier

In [ ]:
# every merger (grey) and the ones detected with probability ∝ chirp mass^K (blue)
selection_controls()

<img src="images/day19_static_selection_widget.png" alt="The selection widget at its opening values: a lognormal population of chirp masses with median 12 as a grey outline, and the detected ones as a blue filled histogram with median 21, shifted to higher mass" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

* the grey outline is a population we make up - a **lognormal** (i.e. $\ln\mathcal{M}$ is Gaussian) with median 12 $M_\odot$
* $p_{\rm det}$, the chance LIGO detects a merger, goes as VT, so we take $p_{\rm det} \propto \mathcal{M}^K$ with $K = 2.2$ ($\mathcal{M} \propto M_{\rm tot}$ at a fixed mass ratio)
* the blue histogram keeps each merger with that probability; slide $K$ to 0 and the two agree

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note the detected median is 21, against 12 for all mergers - with no measurement error at all!**

</div>

## When faint objects outnumber bright ones, scatter pushes more up over a threshold than down - Eddington bias

In [ ]:
# steeply falling counts, measured with scatter, kept above one threshold
eddington_controls()

<img src="images/day19_static_eddington_widget.png" alt="The Eddington widget at its opening values. Left: counts per 0.1 dex of luminosity on a log axis, falling steeply; the true counts in dark grey and the measured counts, with 0.2 dex scatter, in blue, 1.3 times higher; a dashed survey threshold at log L = 1. Right: the true luminosities of the objects kept above the threshold, with the 37 percent that are truly below it in orange; the kept objects are measured 0.14 dex too bright" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget, for readers on GitHub or without running the notebook)

<div style="font-size:0.8em; text-align:center; color:#666;">Our simulation (seed 19): steepness 1.5 means counts fall about $30\times$ for every $10\times$ in luminosity; the survey keeps everything measured above $\log L = 1$.</div>

* faint objects far outnumber bright ones, so scatter moves more up than down, and the measured counts (blue) are above the true ones
* right: of the objects the survey keeps, 37% are truly below the threshold
* slide the scatter to 0 and the bias is gone; make the counts steeper and it grows
* for LIGO, noise can lift a quiet merger over SNR 8

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The objects a survey keeps near its threshold are measured too bright - here by 0.14 dex.**

</div>

## So how do we put the detector into Tuesday's graph?

<img src="images/day19_pgm_pooling_gw.png" alt="Tuesday's partial pooling graph: blue mu and sigma_pop, white true chirp mass theta_j, grey measured chirp mass y_j with its error bar sigma_j, in a plate over the 32 events" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

* the plate runs over the 32 we detected, and nothing in the graph says why these 32 (the new piece is in orange on each of the next three slides)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Ideally, we should include the selection process in our model, i.e. expand it to include the mergers we didn't detect.**

</div>

## We add the distance $d_j$, and the data become $D_j$

<img src="images/day19_pgm_sel_b_distance.png" alt="Step 2: blue mu and sigma_pop point to white theta_j; a new white node d_j, the true distance, outlined in orange; theta_j and d_j both point to a grey data node D_j, also outlined in orange; the plate is over the 32 events" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

* at the same mass, a farther merger is quieter (reach plot), so the true distance $d_j$ (latent, white) gets an arrow into the data
* $y_j \pm \sigma_j$ is a chirp mass that LIGO estimated for us, but LIGO kept event $j$ because of its SNR, which $y_j$ doesn't contain
* so the grey circle becomes $D_j$, the strain the detectors recorded (GW150914's wiggles); $y_j$, $\sigma_j$, the distance and the SNR are all computed from it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A light merger nearby and a heavy one far away can give the same SNR.**

</div>

## The flag $I_j$ records whether an event's data are above SNR 8

<img src="images/day19_pgm_sel_c_detect.png" alt="Step 3: the same graph with a new grey detection flag I_j, outlined in orange, fed by the data D_j and by a fixed dot labelled SNR 8" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

* $I_j = 1$ when the SNR in $D_j$ is above 8 (black dot: fixed, not fit)
* the arrow comes from $D_j$, not $\theta_j$: LIGO decides on what it measured, so noise can lift an event over (Eddington bias)
* averaged over the noise and the distance, the chance that $I_j = 1$ for a merger of true mass $\theta$ is $p_{\rm det}(\theta)$, which goes as VT

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Because $I_j$ hangs off $D_j$, detection goes into the likelihood like any other data.**

</div>

## The plate now runs over every merger, including the ones we missed

<img src="images/day19_pgm_sel_d_missed.png" alt="Step 4: the plate now runs over j = 1 to N; a new blue node N, the total number of mergers, and a new grey node N_det, the number detected, fed by N and by the flags I_j, both outlined in orange" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Grey = data: observed for the 32 we detected, never seen for the ones we missed.</div>

* $N_{\rm det} = 32$ is what we count
* for a missed merger, $\theta_j$, $d_j$ and $D_j$ are all unobserved

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The model must now contain the total number of mergers $N$ (since this is a truncation problem).**

</div>

## We can ignore selection only if it doesn't depend on data we didn't observe

<img src="images/day19_pgm_selection_gw.png" alt="The finished graph with selection: blue mu, sigma_pop and N; inside a plate over all N mergers, white theta_j and d_j point to grey data D_j, which with a fixed SNR 8 dot points to the grey flag I_j; the flags and N point to N_det" style="display:block;margin:0 auto;max-height:220px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">The two conditions are after Mantz &amp; Marshall (2019); the graph here is ours.</div>

* ignoring selection means the posterior that knows about detection equals the one that doesn't
* that holds if selection doesn't depend on data we didn't observe, and the threshold is fixed in advance (ours: SNR 8), so it tells us nothing about $\mu$ and $\sigma_{\rm pop}$

Is LIGO's selection independent of (potentially) unobserved data?

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Heck no.** Detection depends on $D_j$, and a quieter merger (lighter, or farther) is exactly the one whose $D_j$ we never see.

</div>

## In `PyMC`, we add the detector with `pm.Potential`

$$\ln \mathcal{L} = \sum_{j=1}^{N_{\rm det}} \left[\ln p(y_j \mid \theta_j) + \ln p_{\rm det}(\theta_j)\right] - N_{\rm det} \ln\alpha(\mu, \sigma_{\rm pop}), \qquad \alpha = \int p_{\rm det}(\theta)\, p(\theta \mid \mu, \sigma_{\rm pop})\, d\theta$$

```python
with pm.Model():
    mu = pm.Normal('mu', np.log(20), 1)                       # mean of ln(chirp mass)
    sigma_pop = pm.HalfNormal('sigma_pop', 1)
    theta = pm.Normal('theta', mu, sigma_pop, shape=N_det)    # theta_j = ln(chirp mass), centered
    pm.Potential('selection', K * theta.sum() - N_det * (K * mu + K**2 * sigma_pop**2 / 2))
    pm.Normal('obs', theta, mc_err / mc, observed=np.log(mc))
```

* $\alpha$ is the fraction of the population that we'd detect; the missed mergers (and $N$) enter only through it
* $\theta_j$ and $y_j$ are now ln chirp masses, true and measured; $y_j$ is the part of $D_j$ that tells us about $\theta_j$
* errors (9%) are small next to the spread (0.6 in ln), so we write it centered
* with $\mathcal{M}^K = e^{K\theta}$, $\ln\alpha = K\mu + K^2\sigma_{\rm pop}^2/2$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The detector adds $\ln p_{\rm det}$ for each event and subtracts $\ln\alpha$ once per event** (`pm.Potential` adds a term to $\ln p$)

</div>

In [ ]:
# The population of ln chirp mass, fit with and without the detector (a "skip" cell)
# with p_det ∝ Mc^K and a lognormal population, alpha has a closed form: ln alpha = K mu + K^2 sigma_pop^2 / 2
def plot_corrected():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.4, 3.6), gridspec_kw={'width_ratios': [1, 1.4]})
    b = np.linspace(2, 30, 70)
    for key, col, lab in [('ln_naive', 'C1', 'ignoring the detector'), ('ln_sel', 'C2', 'with the detector')]:
        v = np.exp(runs[key]['mu'].ravel())
        a1.hist(v, b, density=True, color=col, alpha=0.6, label=f'{lab}: {np.median(v):.0f}')
    a1.set_xlabel(r'median chirp mass of the population [$M_\odot$]'); a1.set_ylabel('posterior'); a1.set_yticks([]); a1.legend(fontsize=10, loc='upper right')
    a1.set_title('posterior, 4000 draws each', fontsize=12)
    m = np.linspace(1, 80, 400)
    lnpdf = lambda mu, sp: np.exp(-0.5 * ((np.log(m) - mu) / sp)**2) / (m * sp * np.sqrt(2 * np.pi))
    a2.hist(mc, np.linspace(0, 70, 22), density=True, color='0.8', label='the 32 detected')
    for key, col, lab in [('ln_naive', 'C1', 'fit ignoring the detector'), ('ln_sel', 'C2', 'the population, with the detector')]:
        mu, sp = np.median(runs[key]['mu']), np.median(runs[key]['sigma_pop'])
        a2.plot(m, lnpdf(mu, sp), color=col, lw=2, label=lab)
    mu, sp = np.median(runs['ln_sel']['mu']), np.median(runs['ln_sel']['sigma_pop'])
    a2.plot(m, lnpdf(mu + K * sp**2, sp), color='k', ls='--', lw=1.5, label=r'that population $\times\,\mathcal{M}^K$')
    a2.set_xlabel(r'chirp mass [$M_\odot$]'); a2.set_ylabel('density'); a2.set_yticks([]); a2.legend(fontsize=9, loc='upper right')
    plt.tight_layout(); plt.show()

## With the detector in the model, the typical chirp mass is 9 $M_\odot$ (19 without it)

In [ ]:
# left: the population's median chirp mass, without and with the detector; right: the two fitted populations
plot_corrected()

* left: the two posteriors for the population's median chirp mass barely overlap
* right: the corrected population (green), passed back through the detector (dashed), matches the fit that ignored it (orange) - the detected sample is reproduced
* one lognormal (median 9, mode about 6) can't hold both of the population paper's peaks (8.3 and 27.9 $M_\odot$)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Most black-hole mergers are lighter than the ones LIGO detects.**

</div>

## What did we learn today?

* a graph is a model you can write as `PyMC` code, one line per circle
* after every run, check $\hat R$, the effective samples and the divergences
* a divergence means the step size didn't fit the posterior's shape; for weak data, rewrite the model non-centered
* truncation (missing objects) and censoring (missing values)
* Malmquist and Eddington bias make a detected sample brighter than the population
* with the detector in the likelihood, the typical chirp mass of LIGO's black holes is 9 $M_\odot$ (19 without it)
* next week, Nov 3 and 5: no likelihood at all - ABC (approximate Bayesian computation) and simulation-based inference

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If selection can't be ignored, put the detector into the likelihood.**

</div>

## Before you go

* Lab 08 is posted today, due **Wed Nov 4 by Noon**: your midterm's Part 2 as a graph, refit in `PyMC` (every date is in `ASSIGNMENTS.md`)
* **Quiz 4, Tue Nov 3**, start of class (Abha is in the room): Gaussian processes and selection effects
* colloquium Tue Nov 3, 3:45 pm, this room: Rachel Bezanson (Pitt); refreshments 3:30 pm, Astronomy 222
* EXTRA, after the defenses: more on the funnel, LIGO's own selection check, a supernova survey, and upper limits

## Lab 07 defenses, now: Chayil Ewing, Dante Ratchev

* 4 questions each, one per slot (walk-through, a choice, a check, a stretch), about 90 seconds a slot
* everyone else stays and listens

## EXTRA: the funnel, as Betancourt & Girolami draw it

Not presented today.

<img src="images/day19_betancourt_girolami_funnel.png" alt="Betancourt and Girolami 2013 Fig. 2: the funnel distribution in the plane of one parameter theta_i against v, the log of the variance, colored by the potential energy, with arrows showing the local curvature, which changes strongly from the wide top to the narrow bottom" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Betancourt &amp; Girolami 2013, arXiv:1312.0906, Fig. 2. Neal (2003, Annals of Statistics 31, 705) introduced the shape. The color, capital $V$, is the potential energy.</div>

* their $\theta_i$ (our $\theta_j$) against $v$, the log of the variance (not Day 14's velocity); our funnel plot had $\sigma_{\rm pop}$ on a log axis, the same thing
* the arrows show the curvature changing strongly from the top to the neck, which they call "typical of hierarchical models" when the data are sparse

## EXTRA: how LIGO checks its population against its detections

Not presented today.

<img src="images/day19_gwtc3_fig9_selection.png" alt="GWTC-3 population paper Fig. 9: cumulative distributions of the observed primary mass, effective spin and redshift of the detected binary black holes in blue, against the distributions expected to be observed from the earlier population fit after selection, in grey with 90 percent bands" style="display:block;margin:0 auto;max-height:230px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Abbott et al. 2023, PRX 13, 011048, Fig. 9 (CC BY 4.0). Three panels: the heavier mass, the effective spin and the redshift; each curve is the fraction of events below $x$.</div>

* the blue line is the detected events, and the grey band is what the previous catalog's population predicts we'd detect
* the authors find them broadly consistent with what they expected
* the check compares what you'd *detect* with what was detected - pass a population model through the detector before comparing it with data
* our $p_{\rm det} \propto \mathcal{M}^{2.2}$ ignores redshift, spins and mass ratio, and LIGO measures $p_{\rm det}$ by injecting fake signals into the real pipeline
* why VT grows as $M_{\rm tot}^{2.2}$, flatter than our $\mathcal{M}^{5/2}$: heavy mergers end at low frequencies, where the detectors are less sensitive, and at large distances volume grows more slowly than $d^3$ (the universe's expansion)

## EXTRA: a magnitude-limited supernova survey

Not presented today.

<img src="images/day19_march2018_malmquist.png" alt="March et al. 2018 Fig. 1: peak B magnitude against redshift for 250 simulated type Ia supernovae; blue points observed, grey points fainter than m_B = 24 unobserved; a red line for the true cosmology and a blue dotted line for a matter-dominated universe that the cut sample would wrongly favor" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">March, Wolf, Sako, D'Andrea &amp; Brout 2018, arXiv:1804.02474, Fig. 1 (simulated supernovae).</div>

* the grey supernovae are fainter than $m_B = 24$ and never observed, and the blue ones were
* fit the blue points alone and a matter-dominated universe (one with no dark energy) is wrongly favored (their caption)
* your midterm stopped at redshift 0.15, well short of where this bites
* this is Malmquist bias, and the fix is a hierarchical model with the truncation in it

## EXTRA: upper limits

Not presented today.

* Kelly 2007 (ApJ 665, 1489), the LinMix fit behind Tuesday's Reines & Volonteri slide, also fits censored data
* an upper limit enters the likelihood as $\int_{-\infty}^{y_{\rm lim}} p(y \mid x, \theta)\, dy$, the chance the true value is below the limit